In [110]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

import json

# Spotipy Setting

In [111]:
secrets = json.loads(open('secrets.json').read())

client_id = secrets["client_id"]
client_secret= secrets["client_secret"]

client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# PySpark를 사용하여 전처리 진행하기

In [112]:
# MongoDB와 SparkSession 연결하기

spark = SparkSession.builder.master("local[*]").appName("Preprocessing").config("spark.executor.memory", "10g").config('spark.mongodb.input.uri', secrets["mongo_host"])\
.config('spark.mongodb.output.uri', secrets["mongo_host"])\
.config('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1')\
.getOrCreate()

In [113]:
df = spark.read.format("mongo").load()
df.show() # 잘 불러와 지는 것을 확인할 수 있음

+--------------------+----------+----+--------------------+-------+--------------------+
|              Artist|      Date|Rank|               Title|TrackID|                 _id|
+--------------------+----------+----+--------------------+-------+--------------------+
|        Travis Scott|2023-08-28|  97|           THANK GOD|   NULL|{64f426a40a9a83cb...|
|Oliver Anthony Music|2023-08-28|   1|Rich Men North of...|   NULL|{64f426a40a9a83cb...|
|Taylor Swift Feat...|2023-08-12|  20|               Karma|   NULL|{64f426a40a9a83cb...|
|           Jon Pardi|2023-08-20|  58|  Your Heart Or Mine|   NULL|{64f426a40a9a83cb...|
|          Zach Bryan|2023-08-20|  86|  Oklahoma Smokeshow|   NULL|{64f426a40a9a83cb...|
|              Kaliii|2023-08-20|  57|          Area Codes|   NULL|{64f426a40a9a83cb...|
|              DaBaby|2023-08-12|  84|          Shake Sumn|   NULL|{64f426a40a9a83cb...|
|          Young Thug|2023-08-20|  85|Oh U Went (feat. ...|   NULL|{64f426a40a9a83cb...|
|       Lainey Wilson

In [114]:
df.createOrReplaceTempView("PREPROCESSING") # SQL 문법 적용을 위한 TempView 생성

In [115]:
track_id = spark.sql("SELECT TrackID FROM PREPROCESSING WHERE TrackID IS NOT NULL") # TrackID column에서 NULL 값이 아닌 경우만 불러와 spark DataFrame 생성

In [99]:
# 전처리에서 활용할 udf(user define function) 정의를 위해 함수 정의

def track(x):
    import json
    track_info = sp.track(x)
    return json.dumps(track_info)

def title(x):
    import json
    track_info = json.loads(x)
    title = track_info["name"]
    return title

def artist(x):
    import json
    track_info = json.loads(x)
    artist_name = track_info["artists"][0]["name"]
    return artist_name

def followers(x):
    import json
    track_info = json.loads(x)
    artist_id = track_info["artists"][0]["id"]
    artist_info = sp.artist(artist_id)
    return artist_info["followers"]["total"]

def audio_features(x):
    import json
    audio_features = sp.audio_features(x)[0]
    return json.dumps(audio_features)

def danceability(x):
    import json
    audio_features = json.loads(x)
    return audio_features["danceability"]

def energy(x):
    import json
    audio_features = json.loads(x)
    return audio_features["energy"]

def loudness(x):
    import json
    audio_features = json.loads(x)
    return audio_features["loudness"]

def speechiness(x):
    import json
    audio_features = json.loads(x)
    return audio_features["speechiness"]

def acousticness(x):
    import json
    audio_features = json.loads(x)
    return audio_features["acousticness"]

def instrumentalness(x):
    import json
    audio_features = json.loads(x)
    return audio_features["instrumentalness"]

def liveness(x):
    import json
    audio_features = json.loads(x)
    return audio_features["liveness"]

def valence(x):
    import json
    audio_features = json.loads(x)
    return audio_features["valence"]

def tempo(x):
    import json
    audio_features = json.loads(x)
    return audio_features["tempo"]

def duration_ms(x):
    import json
    audio_features = json.loads(x)
    return audio_features["duration_ms"]

In [100]:
# 정의한 함수들을 udf로 변환

udf_track = udf(track)
udf_title = udf(title)
udf_artist = udf(artist)
udf_followers = udf(followers)
udf_audio_features = udf(audio_features)
udf_danceability = udf(danceability)
udf_energy = udf(energy)
udf_loudness = udf(loudness)
udf_speechiness = udf(speechiness)
udf_acousticness = udf(acousticness)
udf_instrumentalness = udf(instrumentalness)
udf_liveness = udf(liveness)
udf_valence = udf(valence)
udf_tempo = udf(tempo)
udf_duration_ms = udf(duration_ms)

In [102]:
# udf를 사용하여 새로운 column을 생성해가며 전처리 진행

from pyspark.sql.functions import lit

new_df = track_id.withColumn("track", udf_track(track_id.TrackID))
new_df = new_df.withColumn("title", udf_title(new_df.track))
new_df = new_df.withColumn("artist", udf_artist(new_df.track))
new_df = new_df.withColumn("followers", udf_followers(new_df.track))
new_df = new_df.withColumn("audio_features", udf_audio_features(track_id.TrackID))
new_df = new_df.withColumn("danceability", udf_danceability(new_df.audio_features))
new_df = new_df.withColumn("energy", udf_energy(new_df.audio_features))
new_df = new_df.withColumn("loudness", udf_loudness(new_df.audio_features))
new_df = new_df.withColumn("speechiness", udf_speechiness(new_df.audio_features))
new_df = new_df.withColumn("acousticness", udf_acousticness(new_df.audio_features))
new_df = new_df.withColumn("instrumentalness", udf_instrumentalness(new_df.audio_features))
new_df = new_df.withColumn("liveness", udf_liveness(new_df.audio_features))
new_df = new_df.withColumn("valence", udf_valence(new_df.audio_features))
new_df = new_df.withColumn("tempo", udf_tempo(new_df.audio_features))
new_df = new_df.withColumn("duration_ms", udf_duration_ms(new_df.audio_features))
new_df = new_df.withColumn("Hit", lit(1))
new_df.show()

+--------------------+--------------------+--------------------+---------------+---------+--------------------+------------+------+--------+-----------+------------+----------------+--------+-------+-------+-----------+---+
|             TrackID|               track|               title|         artist|followers|      audio_features|danceability|energy|loudness|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|duration_ms|Hit|
+--------------------+--------------------+--------------------+---------------+---------+--------------------+------------+------+--------+-----------+------------+----------------+--------+-------+-------+-----------+---+
|1BxfuPKGuaTgP7aM0...|{"album": {"album...|        Cruel Summer|   Taylor Swift|100101203|{"danceability": ...|       0.552| 0.702|  -5.707|      0.157|       0.117|         2.06E-5|   0.105|  0.564|169.994|     178427|  1|
|4xhsWYTOGcal8zt0J...|{"album": {"album...|         Lovin On Me|    Jack Harlow|  3475159|{"danceability